In [ ]:
from openai import OpenAIimport timeimport osfrom dotenv import load_dotenvload_dotenv()client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))MODEL = "gpt-4o-mini"RATE_LIMIT_DELAY = 1.2  # secondsdef simulate_user_reply(conversation_history, chatbot_question):    system_prompt = f"""You are playing the role of a real human user chatting with a support chatbot.This is your conversation so far:{conversation_history}Now the chatbot is trying to understand your intent. It may ask you to choose between two or three possible options, or clarify your issue.Respond like a real person would. Sometimes people are clear and direct and may choose option they find reasonable, sometimes they’re informal or unsure. Use your own words.It's okay to be casual, make typos,or ramble a bit sometimes. Vary your behavior the way real users do.Do not keep repreating same question again and again. Behave like user who wants to make chatbot understand their query.If  chatbot responds with something completely irrelevant to your question inform it so."""    messages = [        {"role": "system", "content": system_prompt},        {"role": "user", "content": chatbot_question}    ]    time.sleep(RATE_LIMIT_DELAY)    response = client.chat.completions.create(        model=MODEL,        messages=messages,        temperature=0.01    )    return response.choices[0].message.content.strip()

In [ ]:

import re
from difflib import SequenceMatcher

STOPWORDS = {
    'the', 'is', 'in', 'at', 'on', 'for', 'to', 'a', 'of', 'and', 'are', 'what',
    'how', 'i', 'it', 'you', 'this', 'that', 'with', 'me', 'do', 'does', 'my',
    'just', 'want', 'need', 'thanks', 'please', 'could', 'would', 'like', 'can'
}

def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r'[^\w\s]', '', text)
    return text.replace('_', ' ')

def remove_stopwords(words):
    return [word for word in words if word not in STOPWORDS]

def is_close_match(response, options, similarity_threshold=0.75, overlap_threshold=0.6):
    response_norm = normalize_text(response)
    response_words = set(remove_stopwords(response_norm.split()))

    best_match = None
    best_score = 0

    for option in options:
        option_norm = normalize_text(option)
        option_words = set(remove_stopwords(option_norm.split()))

        if not option_words:
            continue

        # Overlap score
        overlap = response_words & option_words
        overlap_ratio = len(overlap) / len(option_words)

        # Fuzzy similarity
        fuzzy_ratio = SequenceMatcher(None, response_norm, option_norm).ratio()

        # Total score: prioritize overlap, use fuzzy as tie-breaker
        score = 0
        if overlap_ratio >= overlap_threshold:
            score = 0.7 * overlap_ratio + 0.3 * fuzzy_ratio
        elif fuzzy_ratio >= similarity_threshold:
            score = fuzzy_ratio * 0.6  # Lower weight if only fuzzy match

        if score > best_score:
            best_score = score
            best_match = option

    return best_match
    '''
    return 0
    '''


In [ ]:
!pip install torch --upgrade --force-reinstall


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.7/766.7 MB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.1/150.1 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from difflib import SequenceMatcher
from transformers import AutoModel, AutoTokenizer
import numpy as np
from datasets import load_dataset
# --- Setup ---
dataset = load_dataset("banking77")

# Intent mapping
intent_feature = dataset["train"].features["label"]
# index_to_name example: {0: 'restaurant_reviews', 1: 'nutrition_info', ..., 149: 'last_intent'}
index_to_name = {i: name for i, name in enumerate(intent_feature.names) if name != "oos"}

'''
def boost_similarity_with_keywords(intent, query, similarity_score):
    """
    Boosts the similarity score for an intent based on all matching keywords.

    Args:
        intent (str): The intent to boost.
        query (str): The user query.
        similarity_score (float): The current similarity score.

    Returns:
        float: The boosted similarity score.
    """
    if intent in intent_keywords:
        keywords = intent_keywords[intent]
        for keyword, weight in keywords:
            if keyword.lower() in query.lower():
                # Dynamic boost based on keyword weight
                boost = 0.6 * weight  # Adjust boost based on keyword weight
                similarity_score += boost  # Accumulate boosts for all matching keywords
    return similarity_score  # No need to cap at 1.0 (normalization will handle it)
'''
# Step 1: Load Sentence-BERT model
class SentenceEmbedder:
    def __init__(self):
        # Load a pre-trained Sentence-BERT model
        self.model = SentenceTransformer('intfloat/e5-base')  # Lightweight and effective

    def get_embedding(self, text):
       """
        Generate embedding for a given text, always prepending 'query: '.

        Args:
            text (str): The input string (treated as a query).

        Returns:
            np.ndarray: The embedding vector.
        """
        # Always prepend 'query: ' to the text
       formatted_text = "query: " + text.strip()
       return self.model.encode(formatted_text)

# Step 2: Prepare intent embeddings
class IntentEmbeddings:
    def __init__(self, intents):
        self.intents = intents
        self.embedder = SentenceEmbedder()
        self.intent_embeddings = self.compute_intent_embeddings()

    def compute_intent_embeddings(self):
        """Generate embeddings for all intents."""
        embeddings = {}
        for intent, description in self.intents.items():
            combined_text = f"{intent}: {description}"  # Combine intent name and description
            embeddings[intent] = self.embedder.get_embedding(combined_text)
        return embeddings

# Step 3: Compute Dempster-Shafer Mass Function
class DSMassFunction:
    def __init__(self, intent_embeddings, hierarchy, custom_thresholds=None):
        self.intent_embeddings = intent_embeddings
        self.embedder = SentenceEmbedder()
        self.hierarchy = hierarchy
        self.custom_thresholds = custom_thresholds or {}  # Use custom thresholds if provided
        self.conversation_history=[]
        self.user_response=None
        self.bert_model = trainer.model # Access the model from the trainer object
        self.bert_tokenizer = trainer.tokenizer # Access the tokenizer from the trainer object
        #self.intent_embeddings = {
            #label: None for label in self.bert_model.config.id2label.values()
        #}
        self.id2label = self.bert_model.config.id2label
        self.label2id = self.bert_model.config.label2id
        #self.clf = svm_model
        self.index_to_name = index_to_name
    def is_leaf(self, intent):
        """Check if a node is a leaf in the hierarchy."""
        return len(self.hierarchy.get(intent, [])) == 0

    def get_threshold(self, intent):
        """Get threshold based on the level of the intent in the hierarchy."""
        if self.is_leaf(intent):
            return 0.1  # Minimum threshold for leaf nodes
        elif intent in self.hierarchy:  # Intermediate nodes
            return 0.2
        else:
            return 0.3  # Highest threshold for root nodes

    def get_confidence_threshold(self, intent):
        """Get confidence threshold for an intent. Use custom threshold if available, else default."""
        if intent in self.custom_thresholds:
            return self.custom_thresholds[intent]  # Use custom threshold
        elif self.is_leaf(intent):
            return 0.3  # Default for leaf nodes
        elif intent in self.hierarchy:  # Intermediate nodes
            return 0.5  # Default for intermediate nodes
        else:
            return 0.7  # Default for root nodes

    def get_node_depth(self, node):
        """
        Compute the depth of a node in the hierarchy.
        :param node: The node to compute the depth for.
        :return: The depth of the node (leaf nodes have the highest depth values).
        """
        depth = 0
        current = node
        # Traverse down the hierarchy until a leaf node is reached
        while current in self.hierarchy and self.hierarchy[current]:
            depth += 1
            current = self.hierarchy[current][0]  # Traverse to the first child
        return depth

    def find_lowest_common_ancestor(self, nodes):
        """
        Find the lowest common ancestor (LCA) of the given nodes in the hierarchy.
        :param nodes: List of nodes (intents) to find the LCA for.
        :return: The lowest common ancestor node.
        """
        if not nodes:
            return None

        # Find all ancestors for each node
        ancestors = []
        for node in nodes:
            node_ancestors = set()
            current = node
            while True:
                node_ancestors.add(current)
                parent = next((parent for parent, children in self.hierarchy.items() if current in children), None)
                if parent is None:
                    break
                current = parent
            ancestors.append(node_ancestors)
        # Find the intersection of all ancestors
        common_ancestors = set.intersection(*ancestors)

        # The LCA is the node with the minimum depth in the common ancestors
        lca = None
        min_depth = float('inf')
        for ancestor in common_ancestors:
            depth = self.get_node_depth(ancestor)
            if depth < min_depth:
                min_depth = depth
                lca = ancestor

        return lca

    def print_node_levels(self):
        """Print the level of each node in the hierarchy."""
        #print("Node Levels:")
        for intent in self.hierarchy:
            depth = self.get_node_depth(intent)
            #print(f"{intent}: Level {depth}")
    '''
    def compute_mass_function(self, user_query, exact_match=None):
        """Compute normalized Dempster-Shafer mass function using clf probabilities."""
        self.conversation_history.append(f"User: {user_query}")
        if exact_match:
            mass_function = {intent: 0.0 for intent in self.intent_embeddings.keys()}
            mass_function[exact_match] = 0.9
            mass_function["Uncertainty"] = 0.1
            #print(f"Direct mass adjustment for exact match: {mass_function}")
            return mass_function

        # Get query embedding
        #embedder = SentenceEmbedder()
        query_embedding = self.embedder.get_embedding(user_query)

        # Get probabilities from clf
        probs = self.clf.predict_proba(query_embedding.reshape(1, -1))[0]

        # Initialize mass function
        mass_function = {intent: 0.0 for intent in self.intent_embeddings.keys()}

        # Assign probabilities to dataset intents
        for idx, prob in zip(self.clf.classes_, probs):
            intent_name = intent_map[idx]
            if intent_name and intent_name in mass_function:
                mass_function[intent_name] = prob

        # Normalize masses
        total_mass = sum(mass_function.values())
        if total_mass > 0:
            mass_function = {intent: mass / total_mass for intent, mass in mass_function.items()}
        else:
            # Handle zero mass case (unlikely with clf)
            mass_function = {intent: 0.0 for intent in self.intent_embeddings.keys()}
            mass_function["Uncertainty"] = 1.0

        #print(f"Mass Function: {mass_function}")
        return mass_function
    '''
    def compute_mass_function(self, user_query, exact_match=None):
        """Compute normalized Dempster-Shafer mass function using BERT-based intent probabilities."""
        self.conversation_history.append(f"User: {user_query}")

        if exact_match:
            mass_function = {intent: 0.0 for intent in self.intent_embeddings.keys()}
            mass_function[exact_match] = 0.9
            mass_function["Uncertainty"] = 0.1
            return mass_function

        # Get probabilities from fine-tuned BERT model
        self.bert_model.eval()
        inputs = self.bert_tokenizer(user_query, return_tensors="pt", truncation=True)
        with torch.no_grad():
            outputs = self.bert_model(**inputs)
            logits = outputs.logits
            probs = F.softmax(logits, dim=-1).squeeze().tolist()

        # Create mass function dictionary
        mass_function = {intent: 0.0 for intent in self.intent_embeddings.keys()}

        # Assign BERT probabilities to intents
        for idx, prob in enumerate(probs):
            intent_name = index_to_name[self.bert_model.config.id2label.get(idx)]
            if intent_name in mass_function:
                mass_function[intent_name] = prob

        # Normalize (optional, already normalized via softmax)
        total_mass = sum(mass_function.values())
        if total_mass > 0:
            mass_function = {intent: mass / total_mass for intent, mass in mass_function.items()}
        else:
            mass_function = {intent: 0.0 for intent in self.intent_embeddings.keys()}
            mass_function["Uncertainty"] = 1.0

        return mass_function




    def compute_belief(self, mass_function):
        """Compute belief values for all intents in a bottom-up manner."""
        belief = {}

        def compute_node_belief(intent):
            # Use memoization to avoid recomputation
            if intent in belief:
                return belief[intent]

            # Start with the mass of the current node
            node_belief = mass_function.get(intent, 0)
            #print(f"Initial belief for {intent}: {node_belief}")

            # Add the belief of all child nodes
            children = self.hierarchy.get(intent, [])
            for child in children:
                child_belief = compute_node_belief(child)
                #print(f"Adding belief of {child} ({child_belief}) to {intent}")
                node_belief += child_belief

            # Cache the belief value for this node
            belief[intent] = node_belief
            #print(f"Final belief for {intent}: {node_belief}")
            return node_belief

        # Compute belief for all nodes in the hierarchy
        for intent in self.hierarchy:
            compute_node_belief(intent)

        return belief

    def combine_mass_functions(self, mass1, mass2):
        """Combine two mass functions using Dempster's rule."""
        combined_mass = {}
        conflict = 0

        for a in mass1:
            for b in mass2:
                # Find the highest common descendant (HCD) of a and b
                hcd = self.find_highest_common_descendant(a, b)

                # If HCD exists, use it as the intersection; otherwise, use "Uncertainty"
                intersection = hcd if hcd else "Uncertainty"

                contribution = mass1[a] * mass2[b]

                if intersection == "Uncertainty":
                    conflict += contribution
                else:
                    combined_mass[intersection] = combined_mass.get(intersection, 0) + contribution

        # Normalize to account for conflict
        if conflict < 1:
            for key in combined_mass:
                combined_mass[key] /= (1 - conflict)

        #print(f"Combined Mass Function: {combined_mass}")
        return combined_mass

    def find_highest_common_descendant(self, node1, node2):
        """
        Find the highest common descendant (HCD) of two nodes in the hierarchy.
        :param node1: The first node.
        :param node2: The second node.
        :return: The highest common descendant if it exists, otherwise None.
        """
        # Get all descendants of node1
        descendants1 = self.get_all_descendants(node1)

        # Get all descendants of node2
        descendants2 = self.get_all_descendants(node2)

        # Find the intersection of the two sets of descendants
        common_descendants = descendants1.intersection(descendants2)

        if not common_descendants:
            return None  # No common descendants

        # Find the highest common descendant (the one with the maximum depth)
        hcd = None
        max_depth = -1
        for descendant in common_descendants:
            depth = self.get_node_depth(descendant)
            if depth > max_depth:
                max_depth = depth
                hcd = descendant

        return hcd

    def get_all_descendants(self, node):
        """
        Get all descendants of a node in the hierarchy, including the node itself.
        :param node: The node to find descendants for.
        :return: A set of all descendants of the node, including the node itself.
        """
        descendants = set()
        stack = [node]

        while stack:
            current = stack.pop()
            descendants.add(current)  # Include the current node itself
            children = self.hierarchy.get(current, [])
            for child in children:
                stack.append(child)
        return descendants

    def evaluate_hierarchy(self, mass_function, current_level):
      """Evaluate the hierarchy and return nodes with belief above the threshold."""
      belief = self.compute_belief(mass_function)  # Precompute beliefs
      confident_nodes = []

      # Check all nodes at the current level
      for intent in current_level:
          intent_belief = belief.get(intent, 0)
          confidence_threshold = self.get_confidence_threshold(intent)  # Use custom thresholds

          if intent_belief >= confidence_threshold:
              confident_nodes.append((intent, intent_belief))
      if confident_nodes:
          #print(f"Confident nodes at current level: {confident_nodes}")  # Debugging output

          # Filter leaf nodes
          confident_leaf_nodes = [node for node in confident_nodes if self.is_leaf(node[0])]

          if confident_leaf_nodes:
              #print(f"Confident leaf nodes: {confident_leaf_nodes}")  # Debugging output

              # Check if one leaf node has a significantly higher belief value
              if len(confident_leaf_nodes) > 1:
                  # Sort by belief value in descending order
                  confident_leaf_nodes.sort(key=lambda x: x[1], reverse=True)

                  # Check if the top node's belief is significantly higher than the second
                  top_node, top_belief = confident_leaf_nodes[0]
                  second_node, second_belief = confident_leaf_nodes[1]

                  # Define a threshold for "significantly higher" (e.g., 20% higher)
                  threshold = 0.4  # Adjust as needed
                  if (top_belief - second_belief) / top_belief > threshold:
                      print(f"Top confident leaf node has significantly higher belief: {top_node}")
                      return [(top_node, top_belief)], belief

              # If no single dominant leaf node, proceed with existing logic
              #print("No single dominant leaf node found. Proceeding with existing logic.")

          # If there is only one confident leaf node, return it
          if len(confident_leaf_nodes) == 1:
              print(f"Single confident leaf node found: {confident_leaf_nodes[0][0]}")
              return confident_leaf_nodes, belief

          # If multiple confident leaf nodes, group them by depth
          depth_groups = {}
          for intent, belief_value in confident_nodes:
              depth = self.get_node_depth(intent)
              if depth not in depth_groups:
                  depth_groups[depth] = []
              depth_groups[depth].append((intent, belief_value))

          # Find the subset of nodes at the lowest level (highest depth value)
          lowest_level = max(depth_groups.keys())  # Use max to find the lowest level
          lowest_level_nodes = depth_groups[lowest_level]

          # If there are multiple nodes at the lowest level, check if any are leaves
          if len(lowest_level_nodes) > 1:
              # Check if any of the nodes at the lowest level are leaves
              leaf_nodes = [node for node in lowest_level_nodes if self.is_leaf(node[0])]
              if leaf_nodes:
                  # If there are leaf nodes, return them
                  #print(f"Leaf nodes found at the lowest level: {leaf_nodes}")
                  return leaf_nodes, belief
              else:
                  # If no leaf nodes, find their LCA
                  lca = self.find_lowest_common_ancestor([intent for intent, _ in lowest_level_nodes])
                  if lca:
                      print(f"Multiple confident nodes found at the lowest level. Lowest Common Ancestor: {lca}")
                      return [(lca, belief.get(lca, 0))], belief
          else:
              # If only one node at the lowest level, return it
              #print(f"Single confident node found at the lowest level: {lowest_level_nodes[0][0]}")
              return [lowest_level_nodes[0]], belief

          return confident_nodes, belief

      # Otherwise, move to the next level (parents of the current level nodes)
      next_level = []
      for node in current_level:
          # Find the parent of the current node
          parent = next((parent for parent, children in self.hierarchy.items() if node in children), None)
          if parent and parent not in next_level:  # Avoid duplicates
             next_level.append(parent)

      if not next_level:  # Base case: no more levels to explore
          return [], belief

      return self.evaluate_hierarchy(mass_function, next_level)

    def ask_clarification(self, parent_nodes, belief):
        """Generate clarification queries for ambiguous parent nodes."""
        clarification_queries = []

        for parent, _ in parent_nodes:
            children = self.hierarchy.get(parent, [])
            if len(children) < 4:
                # Directly ask about all children if they are fewer than 4
                clarification_queries.append((parent, children))
            else:
                # Sort children by belief and pick the top 3
                children_with_belief = [(child, belief.get(child, 0)) for child in children]
                children_with_belief.sort(key=lambda x: x[1], reverse=True)
                top_children = [child for child, _ in children_with_belief[:3]]
                clarification_queries.append((parent, top_children))

        return clarification_queries

    def evaluate_with_clarifications(self, initial_mass, depth=0, max_depth=5):
        """Evaluate recursively with clarification queries until a confident leaf is found."""
        if depth >= max_depth:
            return None, None

        def evaluate_from_leaves(current_mass, depth=0, max_depth=5):
            """Recursively evaluate the hierarchy starting from leaf nodes."""
            if depth >= max_depth:
                return None, None
            # Find all leaf nodes
            leaf_nodes = [intent for intent in self.hierarchy if self.is_leaf(intent)]
            #print(f"Leaf nodes: {leaf_nodes}")  # Debugging output

            # Evaluate the hierarchy starting from the leaf nodes
            confident_nodes, belief = self.evaluate_hierarchy(current_mass, leaf_nodes)

            if confident_nodes:
                # Check if there is exactly one confident leaf node
                confident_leaf_nodes = [node for node in confident_nodes if self.is_leaf(node[0])]
                if len(confident_leaf_nodes) == 1:
                    print(f"Single confident leaf node found: {confident_leaf_nodes[0][0]}")
                    return confident_leaf_nodes[0]

                # If multiple confident leaf nodes, find their LCA
                if len(confident_leaf_nodes) > 1:
                    lca = self.find_lowest_common_ancestor([intent for intent, _ in confident_leaf_nodes])
                    if lca:
                        #print(f"Multiple confident leaf nodes found. Lowest Common Ancestor: {lca}")

                        # Ask for clarification on the children of the LCA
                        #print(f"Asking for clarification on the children of the LCA: {lca}")
                        clarification_queries = self.ask_clarification([(lca, belief.get(lca, 0))], belief)

                        #print("Clarification Queries:")
                        for parent, children in clarification_queries:
                            #print(f"Parent: {parent}, Options: {children}")

                            while True:
                                chatbot_question = f"It seems like you're looking for something related to {parent}. Could you clarify which specific thing you're interested in from this category? Here are a few suggestions: ({children}). Does one of these sound like what you're looking for?"
                                self.conversation_history.append(f"Chatbot: {chatbot_question}")
                                self.user_response = simulate_user_reply("\n".join(self.conversation_history), chatbot_question)
                                #self.conversation_history.append(f"User: {self.user_response}")
                                #print(f"LLM:{self.user_response}")
                                # Check for a close match
                                match = is_close_match(self.user_response, children)
                                if match:
                                    #print(f"User selected (close match): {match}")
                                    user_mass = self.compute_mass_function(user_query=None, exact_match=match)
                                    break
                                else:
                                    #print("Response doesn't closely match any option. Computing mass based on similarity.")
                                    user_mass = self.compute_mass_function(user_query=self.user_response)
                                    break

                            # Combine with existing mass function
                            current_mass = self.combine_mass_functions(current_mass, user_mass)

                        # Continue evaluation with the updated mass function
                        return evaluate_from_leaves(current_mass, depth=depth + 1)
                    else:
                      options_no_parent = []
                      # Sort nodes by belief_value in descending order and pick top 3
                      top_nodes = sorted(confident_nodes, key=lambda x: x[1], reverse=True)[:3]
                      # Add only the intents to options_no_parent
                      for intent, belief_value in top_nodes:
                        options_no_parent.append(intent)
                      chatbot_question = f"There are quite a few different things that might match what you're looking for: ({options_no_parent}). Could you clarify a bit more to help me narrow it down?"
                      self.conversation_history.append(f"Chatbot: {chatbot_question}")
                      self.user_response = simulate_user_reply("\n".join(self.conversation_history), chatbot_question)
                      #self.conversation_history.append(f"User: {self.user_response}")
                      #print(f"LLM:{self.user_response}")
                      user_mass = self.compute_mass_function(user_query=self.user_response)
                      current_mass = self.combine_mass_functions(current_mass, user_mass)

                # Ask clarification queries for non-leaf nodes
                non_leaf_nodes = [(intent, belief_value) for intent, belief_value in confident_nodes if not self.is_leaf(intent)]
                clarification_queries = self.ask_clarification(non_leaf_nodes, belief)

                #print("Clarification Queries:")
                for parent, children in clarification_queries:
                    #print(f"Parent: {parent}, Options: {children}")

                    while True:
                        chatbot_question = f"It seems like you're looking for something related to {parent}. Could you clarify which specific thing you're interested in from this category? Here are a few suggestions: ({children}). Does one of these sound like what you're looking for?"
                        self.conversation_history.append(f"Chatbot: {chatbot_question}")
                        self.user_response = simulate_user_reply("\n".join(self.conversation_history), chatbot_question)
                        #self.conversation_history.append(f"User: {self.user_response}")
                        #print(f"LLM:{self.user_response}")
                        # Check for a close match
                        match = is_close_match(self.user_response, children)
                        if match:
                            #print(f"User selected (close match): {match}")
                            user_mass = self.compute_mass_function(user_query=None, exact_match=match)
                            break
                        else:
                            #print("Response doesn't closely match any option. Computing mass based on similarity.")
                            user_mass = self.compute_mass_function(user_query=self.user_response)
                            break

                    # Combine with existing mass function
                    current_mass = self.combine_mass_functions(current_mass, user_mass)
            else:
                chatbot_question = f"I’m not entirely sure what you're asking. Could you rephrase your question a bit? That would help me better understand what you're looking for!"
                self.conversation_history.append(f"Chatbot: {chatbot_question}")
                self.user_response = simulate_user_reply("\n".join(self.conversation_history), chatbot_question)
                #self.conversation_history.append(f"User: {self.user_response}")
                #print(f"LLM:{self.user_response}")
                user_mass = self.compute_mass_function(user_query=self.user_response)
                current_mass = self.combine_mass_functions(current_mass, user_mass)

            # Recursively evaluate with the updated mass function
            return evaluate_from_leaves(current_mass,depth=depth + 1)

        # Start the evaluation from the leaf nodes with the initial mass
        return evaluate_from_leaves(initial_mass,depth=depth)

'''
# Example Usage
def main():
    # Initialize intent embeddings
    intent_embedder = IntentEmbeddings(hierarchical_intents)
    ds_calculator = DSMassFunction(intent_embedder.intent_embeddings, hierarchy, custom_thresholds)

    # Print node levels for debugging

    # User query
    user_query = "what kind of electrical outlet does that country use"

    # Compute initial Dempster-Shafer mass function
    initial_mass = ds_calculator.compute_mass_function(user_query)

    # Evaluate recursively with clarifications
    ds_calculator.evaluate_with_clarifications(initial_mass)
    print(ds_calculator.conversation_history)
'''

def main():
    from datasets import load_dataset
    import pandas as pd

    # Initialize intent embeddings
    intent_embedder = IntentEmbeddings(hierarchical_intents)

    # Load dataset and define index_to_name
    dataset = load_dataset("banking77")
    intent_feature = dataset["train"].features["label"]
    index_to_name = {i: name for i, name in enumerate(intent_feature.names)}

    # Load and filter test data
    test_data = dataset["test"]
    test_data = [{"text": ex["text"], "label": ex["label"]} for ex in test_data]

    # Split into 10 chunks
    total = len(test_data)
    chunk_size = total // 10
    chunks = [test_data[i*chunk_size : (i+1)*chunk_size] for i in range(9)]
    chunks.append(test_data[9*chunk_size:])  # final chunk includes remainder
    print(chunks[0])
    ds_calculator = DSMassFunction(intent_embedder.intent_embeddings, hierarchy, custom_thresholds)
    # Process each chunk
    for idx, val_chunk in enumerate(chunks):
        print(f"\n--- Processing chunk {idx + 1}/10 ({len(val_chunk)} samples) ---")
        results = []
        for ex in val_chunk:
            user_query = ex["text"]
            true_intent_idx = ex["label"]
            true_intent = index_to_name.get(true_intent_idx, "unknown")
            ds_calculator.conversation_history = []
            initial_mass = ds_calculator.compute_mass_function(user_query)
            DS_prediction = ds_calculator.evaluate_with_clarifications(initial_mass)
            results.append({
                "query": user_query,
                "prediction": str(DS_prediction),
                "true_intent": true_intent,
                "interaction": ds_calculator.conversation_history
            })

            print(ds_calculator.conversation_history)

        # Save chunk results
        df = pd.DataFrame(results)
        df.to_csv(f"predictions_test_chunk_{idx}_e5-base_match_temp_prompt_median_banking=0.01_transformer.csv", index=False)
        print(f"Saved chunk {idx} to predictions_test_chunk_{idx}.csv")

if __name__ == "__main__":
    main()


[{'text': 'How do I locate my card?', 'label': 11}, {'text': 'I still have not received my new card, I ordered over a week ago.', 'label': 11}, {'text': 'I ordered a card but it has not arrived. Help please!', 'label': 11}, {'text': 'Is there a way to know when my card will arrive?', 'label': 11}, {'text': 'My card has not arrived yet.', 'label': 11}, {'text': 'When will I get my card?', 'label': 11}, {'text': 'Do you know if there is a tracking number for the new card you sent me?', 'label': 11}, {'text': 'i have not received my card', 'label': 11}, {'text': 'still waiting on that card', 'label': 11}, {'text': 'Is it normal to have to wait over a week for my new card?', 'label': 11}, {'text': 'How do I track my card?', 'label': 11}, {'text': 'How long does a card delivery take?', 'label': 11}, {'text': "I still don't have my card after 2 weeks.  What should I do?", 'label': 11}, {'text': 'still waiting on my new card', 'label': 11}, {'text': 'I am still waiting for my card after 1 wee

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


Streaming output truncated to the last 5000 lines.
['User: Someone has my card number, freeze my account.']
Single confident leaf node found: compromised_card
Single confident leaf node found: compromised_card
['User: I see random purchases to my account, was it hacked?']
Single confident leaf node found: compromised_card
Single confident leaf node found: compromised_card
['User: It seems someone used my card! There are a few transactions from a small town in the middle of nowhere that I definitely have not made! Please prevent them from using it immediately!']
Single confident leaf node found: card_linking
Single confident leaf node found: card_linking
['User: How do I freeze my card using the app?']
Single confident leaf node found: compromised_card
Single confident leaf node found: compromised_card
['User: My card has been compromised']
Single confident leaf node found: compromised_card
Single confident leaf node found: compromised_card
['User: What should I do if I think that someo

In [ ]:
print(ds_calculator.find_lowest_common_ancestor(['card_linking', 'card_arrival', 'order_physical_card']))

card_issues


In [ ]:
import torch.nn.functional as F # Import the functional module with alias 'F'
intent_embedder = IntentEmbeddings(hierarchical_intents)
ds_calculator = DSMassFunction(intent_embedder.intent_embeddings,hierarchy,custom_thresholds)

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


In [ ]:
print(ds_calculator.compute_mass_function(user_query="I transferred some money and it didint arrive"))

{'Card Issues': 0.0, 'Physical Cards': 0.0, 'activate_my_card': 0.0004650347882328512, 'card_about_to_expire': 0.0007651649380243198, 'card_arrival': 0.001618841147357773, 'card_delivery_estimate': 0.00147625833288546, 'card_not_working': 0.00048631952626486695, 'card_swallowed': 0.0002892725579404012, 'get_physical_card': 0.00036819466969331504, 'getting_spare_card': 0.00036507581578209354, 'lost_or_stolen_card': 0.0003064872997390075, 'order_physical_card': 0.0002398059976440405, 'Virtual Cards': 0.0, 'get_disposable_virtual_card': 0.0007968686149567583, 'getting_virtual_card': 0.00026859201545346127, 'virtual_card_not_working': 0.00033695732345542133, 'Card Payments': 0.0, 'Payment Problems': 0.0, 'card_acceptance': 0.0004036835618664575, 'card_linking': 0.0011642358045085707, 'card_payment_fee_charged': 0.001187424689830188, 'card_payment_not_recognised': 0.0016371277832760876, 'card_payment_wrong_exchange_rate': 0.002781350046911497, 'contactless_not_working': 0.000450807787583878

In [ ]:
def compute_accuracy_from_csv():
    import pandas as pd

    # Try reading CSV
    try:
        df = pd.read_csv("predictions_test_chunk_9_e5-base_match_temp_prompt_median_banking=0.01_transformer.csv")
    except FileNotFoundError:
        print("Error: predictions_test_chunk_0_correxted not found")
        return
    except pd.errors.EmptyDataError:
        print("Error: predictions_test_chunk_0_correxted is empty")
        return

    # Verify columns
    expected_columns = ["query", "prediction", "true_intent"]
    if not all(col in df.columns for col in expected_columns):
        print(f"Error: CSV missing required columns. Found: {list(df.columns)}")
        return

    # Track correct matches
    correct = 0
    total = len(df)

    # Debug: print first few rows
    print("First 5 rows of CSV:")
    print(df.head().to_string())
    print()

    # Count unique values for debugging
    predicted_counts = {}
    true_counts = {}

    # Process each row
    for index, row in df.iterrows():
        # Parse prediction manually
        predicted_intent = "unknown"
        pred_str = row["prediction"]
        if isinstance(pred_str, str):
            try:
                # Expect format: ('intent', np.float64(prob))
                # Remove outer parentheses and split
                cleaned = pred_str.strip("()")
                #parts = cleaned.split(", np.float64(")
                parts = cleaned.split(",")
                if len(parts) > 1:
                    intent_part = parts[0].strip().strip("'")
                    predicted_intent = intent_part.lower().strip()
            except Exception as e:
                print(f"Warning: Failed to parse prediction at row {index}: {pred_str}, error: {e}")

        true_intent = str(row["true_intent"]).lower().strip()

        # Update counts
        predicted_counts[predicted_intent] = predicted_counts.get(predicted_intent, 0) + 1
        true_counts[true_intent] = true_counts.get(true_intent, 0) + 1

        # Compare
        is_correct = predicted_intent == true_intent and true_intent != "unknown"
        if is_correct:
            correct += 1

        # Debug: print first 10 and any matches
        if index < 10 or is_correct:
            print(f"Row {index}: Predicted: {predicted_intent}, True: {true_intent}, Match: {is_correct}")

    # Debug: print intent counts
    print("\nPredicted intent counts:")
    for intent, count in sorted(predicted_counts.items()):
        print(f"{intent}: {count}")
    print("\nTrue intent counts:")
    for intent, count in sorted(true_counts.items()):
        print(f"{intent}: {count}")

    # Compute accuracy
    accuracy = correct / total if total > 0 else 0.0
    print(f"\nAccuracy: {accuracy:.4f} ({correct}/{total} correct)")
compute_accuracy_from_csv()

First 5 rows of CSV:
                                                                                              query                               prediction      true_intent                                                                                                 interaction
0                                                       Do I have to pay for exchanging currencies?  ('exchange_charge', 0.8202134050071012)  exchange_charge                                                       ['User: Do I have to pay for exchanging currencies?']
1                                                        Does it cost extra to exchange currencies?  ('exchange_charge', 0.8245854232712359)  exchange_charge                                                        ['User: Does it cost extra to exchange currencies?']
2                                                 Is there a fee for exchanging foreign currencies?     ('exchange_charge', 0.8348344777002)  exchange_charge                        

In [ ]:
import pandas as pd
import re
from sklearn.metrics import precision_recall_fscore_support

def parse_prediction(pred_str):
    if isinstance(pred_str, str):
        try:
            match = re.match(r"\(?\s*'([^']+)'\s*,\s*np\.float64\([^)]+\)\s*\)?", pred_str)
            if match:
                return match.group(1).lower().strip()
        except:
            pass
    return "unknown"

def compute_f1_to_csv():
    # List of known chunk files
    files = [
        "predictions_test_chunk_0_e5-base_banking_temp=0.05.csv",
        "predictions_test_chunk_1_e5-base_banking_temp=0.05.csv",
        "predictions_test_chunk_2_e5-base_banking_temp=0.05.csv",
        "predictions_test_chunk_3_e5-base_banking_temp=0.05.csv",
        "predictions_test_chunk_4_e5-base_banking_temp=0.05.csv",
        "predictions_test_chunk_5_e5-base_banking_temp=0.05.csv",
        "predictions_test_chunk_6_e5-base_banking_temp=0.05.csv",
        "predictions_test_chunk_7_e5-base_banking_temp=0.05.csv",
        "predictions_test_chunk_8_e5-base_banking_temp=0.05.csv",
        "predictions_test_chunk_9_e5-base_banking_temp=0.05.csv"
    ]
    all_true = []
    all_pred = []

    for filename in files:
        try:
            df = pd.read_csv(filename)
        except FileNotFoundError:
            print(f"File not found: {filename}")
            continue
        except pd.errors.EmptyDataError:
            print(f"File is empty: {filename}")
            continue

        if not all(col in df.columns for col in ["prediction", "true_intent"]):
            print(f"Skipping {filename}: missing columns")
            continue

        for _, row in df.iterrows():
            true_intent = str(row["true_intent"]).lower().strip()
            pred_intent = parse_prediction(row["prediction"])
            all_true.append(true_intent)
            all_pred.append(pred_intent)

    # Calculate precision, recall, F1 score, and support
    labels = sorted(set(all_true + all_pred))
    precision, recall, f1, support = precision_recall_fscore_support(
        all_true, all_pred, labels=labels, zero_division=0
    )

    # Create DataFrame
    f1_df = pd.DataFrame({
        "intent": labels,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "support": support
    })

    # Save to CSV
    f1_df.to_csv("f1_scores_per_intent.csv", index=False)
    print("Saved F1 scores to f1_scores_per_intent.csv")

compute_f1_to_csv()


Saved F1 scores to f1_scores_per_intent.csv


In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 16.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is

In [ ]:
hierarchical_intents = {
    "Card Issues": "Issues related to physical or virtual cards and their functionality.",
    "Physical Cards": "Concerns involving physical debit or credit cards.",
    "activate_my_card": "Help with activating a new physical card.",
    "card_about_to_expire": "What to do when your card is about to expire.",
    "card_arrival": "Information about expected delivery of your card.",
    "card_delivery_estimate": "Estimate of how long it takes for your card to arrive.",
    "card_not_working": "Troubleshooting issues with card functionality.",
    "card_swallowed": "What to do if an ATM swallows your card.",
    "get_physical_card": "Requesting a physical card for your account.",
    "getting_spare_card": "Getting a replacement or additional physical card.",
    "lost_or_stolen_card": "Report and handle lost or stolen physical cards.",
    "order_physical_card": "Ordering a new or replacement physical card.",
    "Virtual Cards": "Support for virtual or disposable cards.",
    "get_disposable_virtual_card": "Getting a disposable virtual card.",
    "getting_virtual_card": "How to request or access a virtual card.",
    "virtual_card_not_working": "Problems with using virtual cards.",

    "Card Payments": "Help with payments made using a card.",
    "Payment Problems": "Issues like declined transactions or incorrect fees.",
    "card_acceptance": "Merchant declined card or doesn't accept it.",
    "card_linking": "Linking your card to apps or accounts.",
    "card_payment_fee_charged": "Unexpected fees on card payments.",
    "card_payment_not_recognised": "Disputing an unknown card payment.",
    "card_payment_wrong_exchange_rate": "Incorrect currency exchange rate applied.",
    "contactless_not_working": "Problems with using contactless feature.",
    "declined_card_payment": "Card payment was declined unexpectedly.",
    "disposable_card_limits": "Limits on how disposable cards can be used.",
    "pending_card_payment": "Card transaction pending or delayed.",
    "reverted_card_payment?": "A payment was reverted or refunded.",
    "direct_debit_payment_not_recognised": "Unknown or suspicious direct debit.",
    "transaction_charged_twice": "Same transaction charged multiple times.",

    "Payments and Transfers": "Issues with money transfers and withdrawals.",
    "Transfers": "Managing bank transfers and their statuses.",
    "beneficiary_not_allowed": "Cannot send money to selected beneficiary.",
    "cancel_transfer": "How to cancel a bank transfer.",
    "declined_transfer": "Transfer was declined or failed.",
    "failed_transfer": "Transfer did not complete successfully.",
    "pending_transfer": "Transfer is pending or in process.",
    "transfer_fee_charged": "Unexpected transfer fees.",
    "transfer_into_account": "Receiving transfers into your account.",
    "transfer_not_received_by_recipient": "Recipient hasn't received the money.",
    "transfer_timing": "How long transfers usually take.",
    "Cash Withdrawals": "Withdrawing cash and related issues.",
    "atm_support": "Using ATMs and their availability.",
    "cash_withdrawal_charge": "Fees when withdrawing money.",
    "cash_withdrawal_not_recognised": "Unknown or suspicious cash withdrawal.",
    "declined_cash_withdrawal": "ATM withdrawal was declined.",
    "pending_cash_withdrawal": "Withdrawal is still processing.",
    "wrong_amount_of_cash_received": "Incorrect cash dispensed by ATM.",
    "wrong_exchange_rate_for_cash_withdrawal": "Poor currency conversion on withdrawal.",

    "Account Services": "Managing your account, top-ups, and profile.",
    "Account Management": "Profile updates and account access issues.",
    "balance_not_updated_after_bank_transfer": "Transfer completed but balance not updated.",
    "balance_not_updated_after_cheque_or_cash_deposit": "Cash/cheque deposit not reflected.",
    "edit_personal_details": "Update your name, address, etc.",
    "passcode_forgotten": "What to do if you forget your passcode.",
    "pin_blocked": "Your PIN is blocked — how to unblock or reset.",
    "terminate_account": "Close your account permanently.",
    "Top-Ups": "Adding money to your account.",
    "automatic_top_up": "Enable or troubleshoot automatic top-ups.",
    "pending_top_up": "Top-up is delayed or pending.",
    "top_up_by_bank_transfer_charge": "Charges for top-up by bank.",
    "top_up_by_card_charge": "Fees for top-up by debit/credit card.",
    "top_up_by_cash_or_cheque": "Add funds via cash or cheque.",
    "top_up_failed": "Top-up attempt failed.",
    "top_up_limits": "Limits on how much you can top up.",
    "top_up_reverted": "Top-up got reverted or cancelled.",
    "topping_up_by_card": "How to top up using a card.",
    "verify_top_up": "Verification required for a top-up.",

    "Security and Verification": "ID verification and security concerns.",
    "Verification": "Identity and source of funds checks.",
    "unable_to_verify_identity": "Identity could not be verified.",
    "verify_my_identity": "Steps to verify your identity.",
    "verify_source_of_funds": "Proof for where your money comes from.",
    "why_verify_identity": "Why verification is necessary.",
    "Security Issues": "Card or phone compromised.",
    "compromised_card": "Your card might have been compromised.",
    "lost_or_stolen_phone": "What to do if your phone is lost or stolen.",

    "Fees and Refunds": "Fee details and refund requests.",
    "Charges and Fees": "Unexpected charges and fees.",
    "exchange_charge": "Fee applied during currency exchange.",
    "extra_charge_on_statement": "Extra or unclear charges in statement.",
    "Refunds": "Issues with getting your money back.",
    "Refund_not_showing_up": "A refund hasn’t appeared in your account.",
    "request_refund": "How to request a refund.",

    "General": "General questions not tied to a transaction.",
    "General Inquiries": "Common questions about the service.",
    "age_limit": "Minimum age required to open an account.",
    "apple_pay_or_google_pay": "Using Apple Pay or Google Pay.",
    "change_pin": "How to change your card's PIN.",
    "country_support": "Supported countries for services.",
    "exchange_rate": "Current rates and how they apply.",
    "exchange_via_app": "How to convert currency within the app.",
    "fiat_currency_support": "Which fiat currencies are supported.",
    "receiving_money": "Can you receive money from others?",
    "supported_cards_and_currencies": "List of cards/currencies supported.",
    "visa_or_mastercard": "Card type — Visa or Mastercard."
}




In [ ]:
hierarchy = {
    "Card Issues": [
        "Physical Cards",
        "Virtual Cards"
    ],
    "Physical Cards": [
        "activate_my_card",
        "card_about_to_expire",
        "card_arrival",
        "card_delivery_estimate",
        "card_not_working",
        "card_swallowed",
        "get_physical_card",
        "getting_spare_card",
        "lost_or_stolen_card",
        "order_physical_card"
    ],
    "activate_my_card": [],
    "card_about_to_expire": [],
    "card_arrival": [],
    "card_delivery_estimate": [],
    "card_not_working": [],
    "card_swallowed": [],
    "get_physical_card": [],
    "getting_spare_card": [],
    "lost_or_stolen_card": [],
    "order_physical_card": [],
    "Virtual Cards": [
        "get_disposable_virtual_card",
        "getting_virtual_card",
        "virtual_card_not_working"
    ],
    "get_disposable_virtual_card": [],
    "getting_virtual_card": [],
    "virtual_card_not_working": [],

    "Card Payments": [
        "Payment Problems"
    ],
    "Payment Problems": [
        "card_acceptance",
        "card_linking",
        "card_payment_fee_charged",
        "card_payment_not_recognised",
        "card_payment_wrong_exchange_rate",
        "contactless_not_working",
        "declined_card_payment",
        "disposable_card_limits",
        "pending_card_payment",
        "reverted_card_payment?",
        "direct_debit_payment_not_recognised",
        "transaction_charged_twice"
    ],
    "card_acceptance": [],
    "card_linking": [],
    "card_payment_fee_charged": [],
    "card_payment_not_recognised": [],
    "card_payment_wrong_exchange_rate": [],
    "contactless_not_working": [],
    "declined_card_payment": [],
    "disposable_card_limits": [],
    "pending_card_payment": [],
    "reverted_card_payment?": [],
    "direct_debit_payment_not_recognised": [],
    "transaction_charged_twice": [],

    "Payments and Transfers": [
        "Transfers",
        "Cash Withdrawals"
    ],
    "Transfers": [
        "beneficiary_not_allowed",
        "cancel_transfer",
        "declined_transfer",
        "failed_transfer",
        "pending_transfer",
        "transfer_fee_charged",
        "transfer_into_account",
        "transfer_not_received_by_recipient",
        "transfer_timing"
    ],
    "beneficiary_not_allowed": [],
    "cancel_transfer": [],
    "declined_transfer": [],
    "failed_transfer": [],
    "pending_transfer": [],
    "transfer_fee_charged": [],
    "transfer_into_account": [],
    "transfer_not_received_by_recipient": [],
    "transfer_timing": [],
    "Cash Withdrawals": [
        "atm_support",
        "cash_withdrawal_charge",
        "cash_withdrawal_not_recognised",
        "declined_cash_withdrawal",
        "pending_cash_withdrawal",
        "wrong_amount_of_cash_received",
        "wrong_exchange_rate_for_cash_withdrawal"
    ],
    "atm_support": [],
    "cash_withdrawal_charge": [],
    "cash_withdrawal_not_recognised": [],
    "declined_cash_withdrawal": [],
    "pending_cash_withdrawal": [],
    "wrong_amount_of_cash_received": [],
    "wrong_exchange_rate_for_cash_withdrawal": [],

    "Account Services": [
        "Account Management",
        "Top-Ups"
    ],
    "Account Management": [
        "balance_not_updated_after_bank_transfer",
        "balance_not_updated_after_cheque_or_cash_deposit",
        "edit_personal_details",
        "passcode_forgotten",
        "pin_blocked",
        "terminate_account"
    ],
    "balance_not_updated_after_bank_transfer": [],
    "balance_not_updated_after_cheque_or_cash_deposit": [],
    "edit_personal_details": [],
    "passcode_forgotten": [],
    "pin_blocked": [],
    "terminate_account": [],
    "Top-Ups": [
        "automatic_top_up",
        "pending_top_up",
        "top_up_by_bank_transfer_charge",
        "top_up_by_card_charge",
        "top_up_by_cash_or_cheque",
        "top_up_failed",
        "top_up_limits",
        "top_up_reverted",
        "topping_up_by_card",
        "verify_top_up"
    ],
    "automatic_top_up": [],
    "pending_top_up": [],
    "top_up_by_bank_transfer_charge": [],
    "top_up_by_card_charge": [],
    "top_up_by_cash_or_cheque": [],
    "top_up_failed": [],
    "top_up_limits": [],
    "top_up_reverted": [],
    "topping_up_by_card": [],
    "verify_top_up": [],

    "Security and Verification": [
        "Verification",
        "Security Issues"
    ],
    "Verification": [
        "unable_to_verify_identity",
        "verify_my_identity",
        "verify_source_of_funds",
        "why_verify_identity"
    ],
    "unable_to_verify_identity": [],
    "verify_my_identity": [],
    "verify_source_of_funds": [],
    "why_verify_identity": [],
    "Security Issues": [
        "compromised_card",
        "lost_or_stolen_phone"
    ],
    "compromised_card": [],
    "lost_or_stolen_phone": [],

    "Fees and Refunds": [
        "Charges and Fees",
        "Refunds"
    ],
    "Charges and Fees": [
        "exchange_charge",
        "extra_charge_on_statement"
    ],
    "exchange_charge": [],
    "extra_charge_on_statement": [],
    "Refunds": [
        "Refund_not_showing_up",
        "request_refund"
    ],
    "Refund_not_showing_up": [],
    "request_refund": [],

    "General": [
        "General Inquiries"
    ],
    "General Inquiries": [
        "age_limit",
        "apple_pay_or_google_pay",
        "change_pin",
        "country_support",
        "exchange_rate",
        "exchange_via_app",
        "fiat_currency_support",
        "receiving_money",
        "supported_cards_and_currencies",
        "visa_or_mastercard"
    ],
    "age_limit": [],
    "apple_pay_or_google_pay": [],
    "change_pin": [],
    "country_support": [],
    "exchange_rate": [],
    "exchange_via_app": [],
    "fiat_currency_support": [],
    "receiving_money": [],
    "supported_cards_and_currencies": [],
    "visa_or_mastercard": []
}


In [ ]:
# !pip install datasets
from datasets import load_dataset
from pathlib import Path
import json

# --- Load the Banking77 dataset ---
ds = load_dataset("banking77")
dataset = ds["train"]  # Use train split directly (Banking77 has train/test splits)

# --- Create intent map ---
# Banking77 provides intent names in features["label"].names, no external file needed
intent_names = ds["train"].features["label"].names
intent_map = {i: name for i, name in enumerate(intent_names)}  # Map index to intent name

# Optional: Generate label_map.txt to mimic CLINC150 format (if needed)
def save_intent_map(intent_map, filepath: Path):
    with open(filepath, "w") as f:
        for idx, intent in intent_map.items():
            # Mimic CLINC150 format: index\tbanking:intent
            f.write(f"{idx}\tbanking:{intent}\n")

# Save label_map.txt (uncomment if you need the file)
# save_intent_map(intent_map, Path("label_map.txt"))

# --- Load intent map (optional, only if using external file) ---
def load_intent_map(filepath: Path):
    intent_map = {}
    with open(filepath, "r") as f:
        for line in f:
            idx, label = line.strip().split("\t")
            domain, intent = label.split(":")
            intent_map[int(idx)] = intent  # Only use the intent text
    return intent_map

# Use built-in intent_map; uncomment below to load from file instead
# intent_map = load_intent_map(Path("label_map.txt"))

# --- Filter and reformat the dataset ---
# Banking77 train split is already training data, no split column to filter
train_data = dataset

# Convert to desired format: {"Example": ..., "Label": ...}
formatted_data = [{"Example": ex["text"], "Label": intent_map[ex["label"]]} for ex in train_data]

# --- Print a few samples ---
from pprint import pprint
pprint(formatted_data[:5])

[{'Example': 'I am still waiting on my card?', 'Label': 'card_arrival'},
 {'Example': "What can I do if my card still hasn't arrived after 2 weeks?",
  'Label': 'card_arrival'},
 {'Example': 'I have been waiting over a week. Is the card still coming?',
  'Label': 'card_arrival'},
 {'Example': 'Can I track my card while it is in the process of delivery?',
  'Label': 'card_arrival'},
 {'Example': 'How do I know if I will get my card, or if it is lost?',
  'Label': 'card_arrival'}]


In [ ]:
custom_thresholds={'Account Management': 0.52, 'Account Services': 0.43, 'Card Issues': 0.4, 'Card Payments': 0.35000000000000003, 'Cash Withdrawals': 0.37, 'Charges and Fees': 0.22, 'Fees and Refunds': 0.24, 'General': 0.45, 'General Inquiries': 0.45, 'Payment Problems': 0.35000000000000003, 'Payments and Transfers': 0.4, 'Physical Cards': 0.33, 'Refund_not_showing_up': 0.14, 'Refunds': 0.23, 'Security Issues': 0.26, 'Security and Verification': 0.28, 'Top-Ups': 0.21, 'Transfers': 0.37, 'Verification': 0.22, 'Virtual Cards': 0.44, 'activate_my_card': 0.21, 'age_limit': 0.23, 'apple_pay_or_google_pay': 0.08, 'atm_support': 0.22, 'automatic_top_up': 0.09, 'balance_not_updated_after_bank_transfer': 0.4, 'balance_not_updated_after_cheque_or_cash_deposit': 0.54, 'beneficiary_not_allowed': 0.23, 'cancel_transfer': 0.75, 'card_about_to_expire': 0.22, 'card_acceptance': 0.06, 'card_arrival': 0.53, 'card_delivery_estimate': 0.22, 'card_linking': 0.41000000000000003, 'card_not_working': 0.16, 'card_payment_fee_charged': 0.44, 'card_payment_not_recognised': 0.43, 'card_payment_wrong_exchange_rate': 0.54, 'card_swallowed': 0.1, 'cash_withdrawal_charge': 0.31, 'cash_withdrawal_not_recognised': 0.34, 'change_pin': 0.7000000000000001, 'compromised_card': 0.26, 'contactless_not_working': 0.06, 'country_support': 0.28, 'declined_card_payment': 0.47000000000000003, 'declined_cash_withdrawal': 0.38, 'declined_transfer': 0.26, 'direct_debit_payment_not_recognised': 0.37, 'disposable_card_limits': 0.38, 'edit_personal_details': 0.19, 'exchange_charge': 0.18, 'exchange_rate': 0.31, 'exchange_via_app': 0.45, 'extra_charge_on_statement': 0.39, 'failed_transfer': 0.44, 'fiat_currency_support': 0.6, 'get_disposable_virtual_card': 0.45, 'get_physical_card': 0.39, 'getting_spare_card': 0.51, 'getting_virtual_card': 0.6900000000000001, 'lost_or_stolen_card': 0.29, 'lost_or_stolen_phone': 0.22, 'order_physical_card': 0.23, 'passcode_forgotten': 0.28, 'pending_card_payment': 0.6, 'pending_cash_withdrawal': 0.31, 'pending_top_up': 0.31, 'pending_transfer': 0.25, 'pin_blocked': 0.17, 'receiving_money': 0.13, 'request_refund': 0.35000000000000003, 'reverted_card_payment?': 0.18, 'supported_cards_and_currencies': 0.42, 'terminate_account': 0.35000000000000003, 'top_up_by_bank_transfer_charge': 0.16, 'top_up_by_card_charge': 0.6, 'top_up_by_cash_or_cheque': 0.16, 'top_up_failed': 0.39, 'top_up_limits': 0.24, 'top_up_reverted': 0.19, 'topping_up_by_card': 0.38, 'transaction_charged_twice': 0.11, 'transfer_fee_charged': 0.32, 'transfer_into_account': 0.37, 'transfer_not_received_by_recipient': 0.32, 'transfer_timing': 0.54, 'unable_to_verify_identity': 0.27, 'verify_my_identity': 0.41000000000000003, 'verify_source_of_funds': 0.16, 'verify_top_up': 0.12, 'virtual_card_not_working': 0.08, 'visa_or_mastercard': 0.1, 'why_verify_identity': 0.33, 'wrong_amount_of_cash_received': 0.21, 'wrong_exchange_rate_for_cash_withdrawal': 0.22}



In [ ]:
import pickle
with open('banking77_regression_model_e5-base_0.877.pkl', 'rb') as f:
    clf = pickle.load(f)

In [ ]:
!pip install tqdm

In [ ]:
from tqdm import tqdm


In [ ]:
# Install required packages (if not already installed)
# !pip install datasets pandas numpy tqdm

from datasets import load_dataset
import pandas as pd
import numpy as np
from tqdm import tqdm
import json
import os



# --- Load Banking77 dataset ---
ds = load_dataset("banking77")
dataset = ds["train"]  # Use train split (10,003 examples)

# Override intent_names with correct intents
intent_names  = ds["train"].features["label"].names

print("Corrected Banking77 intent_names:", sorted(intent_names))

# Create intent map with corrected intents
intent_map = {i: name for i, name in enumerate(intent_names)}

# Format data, mapping labels to corrected intents
formatted_data = []
for ex in dataset:
    label = ex["label"]
    # Ensure label is within valid range (0–76)
    if 0 <= label < len(intent_names):
        formatted_data.append({"Example": ex["text"], "Label": intent_map[label]})
    else:
        print(f"Skipping invalid label {label} for query: {ex['text']}")

print(f"Generated formatted_data with {len(formatted_data)} examples")

# Verify formatted_data intents
formatted_intents = set(ex["Label"] for ex in formatted_data)
invalid_intents = formatted_intents - set(intent_names)
if invalid_intents:
    print(f"Error: Invalid intents in formatted_data: {invalid_intents}")
    formatted_data = [ex for ex in formatted_data if ex["Label"] in intent_names]
    print(f"Filtered formatted_data to {len(formatted_data)} examples with valid intents")


# --- Load hierarchy ---


# Validate hierarchy intents against corrected intent_names
hierarchy_intents = []
for category, intents in hierarchy.items():
    if isinstance(intents, list):
        hierarchy_intents.extend(intents)
hierarchy_intents_set = set(hierarchy_intents)
formatted_intents_set = set(ex["Label"] for ex in formatted_data)

# Check for mismatches
if hierarchy_intents_set != formatted_intents_set:
    print("Warning: Intent mismatch between hierarchy and formatted_data")
    print("Intents in hierarchy but not in data:", hierarchy_intents_set - formatted_intents_set)
    print("Intents in data but not in hierarchy:", formatted_intents_set - hierarchy_intents_set)
    formatted_data = [ex for ex in formatted_data if ex["Label"] in hierarchy_intents_set]
    print(f"Filtered formatted_data to {len(formatted_data)} examples with valid intents")
else:
    print("Hierarchy and formatted_data intents match perfectly!")

# --- Function to split a list into n equal parts ---
def chunk_data(data, n_chunks):
    avg_chunk_size = len(data) // n_chunks
    chunks = []
    for i in range(n_chunks):
        start_idx = i * avg_chunk_size
        end_idx = (i + 1) * avg_chunk_size if i != n_chunks - 1 else len(data)
        chunks.append(data[start_idx:end_idx])
    return chunks

# --- Initialize embedder and calculator ---
try:
    intent_embedder = IntentEmbeddings(hierarchy)
    print("IntentEmbedder intents:", sorted(intent_embedder.intent_embeddings.keys()))
except AttributeError:
    print("Warning: Could not access intent_embedder.intent_embeddings. Ensure it's initialized correctly.")
    intent_embedder = IntentEmbeddings(hierarchy)

# Initialize DSMassFunction with hierarchy intents to avoid corrupted intent_names
ds_calculator = DSMassFunction(intent_embedder.intent_embeddings, hierarchy)

# --- Create a dictionary to store belief values for all examples ---
belief_results = []

# --- Split the data into 4 parts ---
n_chunks = 4
chunks = chunk_data(formatted_data, n_chunks)

# --- Process each chunk of data ---
for i, chunk in enumerate(chunks):
    print(f"Processing chunk {i + 1}...")

    chunk_belief_results = []

    for row in tqdm(chunk, desc=f"Chunk {i + 1}", unit="row"):
        user_query = row["Example"]
        correct_intent = str(row["Label"])  # Ensure Python string
        # Safety check for invalid intents
        if correct_intent not in intent_names:
            print(f"Skipping query '{user_query}' with invalid intent '{correct_intent}'")
            continue

        try:
            mass_function = ds_calculator.compute_mass_function(user_query)
            if mass_function is None:
                print(f"Warning: No mass function for query: {user_query}")
                continue

            belief = ds_calculator.compute_belief(mass_function)

            if belief:
                chunk_belief_results.append({
                    "user_question": user_query,
                    "correct_intent": correct_intent,
                    "belief_values": belief
                })
            else:
                print(f"Warning: No belief values for query: {user_query}")

        except Exception as e:
            print(f"Error processing query '{user_query}': {e}")
            print(f"Correct intent: {correct_intent}")
            print("Skipping query to continue processing...")

    # --- Convert the chunk results to a DataFrame ---
    belief_df_chunk = pd.DataFrame(chunk_belief_results)

    # --- Save the results to a CSV file ---
    output_filename = f"e5-base_train_banking_trial_XGB_chunk_{i + 1}.csv"
    belief_df_chunk.to_csv(output_filename, index=False)

    print(f"Chunk {i + 1} saved to {output_filename}")

print("All chunks processed and saved successfully!")

Corrected Banking77 intent_names: ['Refund_not_showing_up', 'activate_my_card', 'age_limit', 'apple_pay_or_google_pay', 'atm_support', 'automatic_top_up', 'balance_not_updated_after_bank_transfer', 'balance_not_updated_after_cheque_or_cash_deposit', 'beneficiary_not_allowed', 'cancel_transfer', 'card_about_to_expire', 'card_acceptance', 'card_arrival', 'card_delivery_estimate', 'card_linking', 'card_not_working', 'card_payment_fee_charged', 'card_payment_not_recognised', 'card_payment_wrong_exchange_rate', 'card_swallowed', 'cash_withdrawal_charge', 'cash_withdrawal_not_recognised', 'change_pin', 'compromised_card', 'contactless_not_working', 'country_support', 'declined_card_payment', 'declined_cash_withdrawal', 'declined_transfer', 'direct_debit_payment_not_recognised', 'disposable_card_limits', 'edit_personal_details', 'exchange_charge', 'exchange_rate', 'exchange_via_app', 'extra_charge_on_statement', 'failed_transfer', 'fiat_currency_support', 'get_disposable_virtual_card', 'get_p

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


Processing chunk 1...


Chunk 1: 100%|██████████| 2500/2500 [05:02<00:00,  8.27row/s]


Chunk 1 saved to e5-base_train_banking_trial_XGB_chunk_1.csv
Processing chunk 2...


...

Chunk 2 saved to e5-base_train_banking_trial_XGB_chunk_2.csv
Processing chunk 3...


Chunk 3: 100%|██████████| 2500/2500 [04:54<00:00,  8.49row/s]


Chunk 3 saved to e5-base_train_banking_trial_XGB_chunk_3.csv
Processing chunk 4...


Chunk 4: 100%|██████████| 2503/2503 [04:38<00:00,  8.97row/s]


Chunk 4 saved to e5-base_train_banking_trial_XGB_chunk_4.csv
All chunks processed and saved successfully!


In [ ]:
import pandas as pd

# File names for the chunks
chunk_files = [
    "e5-base_train_banking_trial_XGB_chunk_1.csv",
    "e5-base_train_banking_trial_XGB_chunk_2.csv",
    "e5-base_train_banking_trial_XGB_chunk_3.csv",
    "e5-base_train_banking_trial_XGB_chunk_4.csv"
]

# Load and concatenate all chunks
dfs = [pd.read_csv(file) for file in chunk_files]
merged_df = pd.concat(dfs, ignore_index=True)

# Save merged file (optional but useful)
merged_df.to_csv("e5-base_train_banking_trial_XGB.csv", index=False)

print("Merged all chunks into _clinc_trial.csv")


Merged all chunks into _clinc_trial.csv


In [ ]:
import pandas as pd
import re

# Load merged CSV
df = pd.read_csv("e5-base_train_banking_trial_XGB.csv")

def parse_belief_values(raw_str):
    pattern = r"'(.*?)': (?:np\.float64\()?([0-9.eE+-]+)\)?"
    matches = re.findall(pattern, raw_str)
    return {intent: float(val) for intent, val in matches}

# Parse the belief values safely
df["belief_values"] = df["belief_values"].apply(parse_belief_values)

# Convert belief_values dicts to a new DataFrame all at once
belief_df = df["belief_values"].apply(pd.Series).fillna(0.0)

# Merge into original and drop the raw column
df = pd.concat([df.drop(columns=["belief_values"]), belief_df], axis=1)

# Save the transformed DataFrame
df.to_csv("transformed_e5-base_train_banking_trial_XGB.csv", index=False)

# Optional preview
print(df.head())


                                       user_question correct_intent  \
0                     I am still waiting on my card?   card_arrival   
1  What can I do if my card still hasn't arrived ...   card_arrival   
2  I have been waiting over a week. Is the card s...   card_arrival   
3  Can I track my card while it is in the process...   card_arrival   
4  How do I know if I will get my card, or if it ...   card_arrival   

   activate_my_card  card_about_to_expire  card_arrival  \
0          0.001528              0.001989      0.883417   
1          0.001406              0.002514      0.879794   
2          0.001401              0.003467      0.864265   
3          0.001720              0.002178      0.884329   
4          0.005147              0.006411      0.530503   

   card_delivery_estimate  card_not_working  card_swallowed  \
0                0.011511          0.008209        0.003444   
1                0.011035          0.007817        0.003088   
2                0.023012    

In [ ]:
import pandas as pd
import ast  # For safely evaluating strings as Python expressions

# Load the CSV file
df = pd.read_csv("train_clinc_trial.csv")

# Parse the 'belief_values' column into a dictionary
df["belief_values"] = df["belief_values"].apply(ast.literal_eval)

# Extract all unique intents from the belief_values dictionaries
all_intents = set()
for belief_dict in df["belief_values"]:
    all_intents.update(belief_dict.keys())

# Create new columns for each intent's belief value
for intent in all_intents:
    df[intent] = df["belief_values"].apply(lambda x: x.get(intent, 0))  # Default to 0 if intent not found

# Drop the original 'belief_values' column (optional)
df = df.drop(columns=["belief_values"])

# Save the transformed data to a new CSV file
df.to_csv("transformed_train_clinc_trial.csv", index=False)

# Display the transformed DataFrame
print(df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'train_clinc_trial.csv'

In [ ]:
import pandas as pd

# Load the original CSV
df = pd.read_csv("transformed_e5-base_train_banking_trial_XGB.csv")

# Define or load hierarchy
# Example:
# with open("hierarchy.json") as f:
#     hierarchy = json.load(f)

# Get all unique intents
intents = set(hierarchy.keys())
for children in hierarchy.values():
    intents.update(children)
intents = sorted(intents)

# Function to get ancestors
def get_ancestors(intent, hierarchy):
    ancestors = set()
    stack = [intent]
    while stack:
        current = stack.pop()
        ancestors.add(current)
        for parent, children in hierarchy.items():
            if current in children:
                stack.append(parent)
    return ancestors

# Create a list of dictionaries for new columns
new_columns = []
for index, row in df.iterrows():
    correct_intent = row["correct_intent"]
    ancestors = get_ancestors(correct_intent, hierarchy)
    new_row = {f"is_correct_{intent}": int(intent in ancestors) for intent in intents}
    new_columns.append(new_row)

from collections import defaultdict

def get_descendants(intent, hierarchy):
    children = hierarchy.get(intent, [])
    all_descendants = set(children)
    for child in children:
        all_descendants.update(get_descendants(child, hierarchy))
    return all_descendants

# Step 2: For each ancestor, sum the values of its descendant columns
for ancestor in hierarchy.keys():
    descendants = get_descendants(ancestor, hierarchy)
    # Only include descendants that actually exist as columns in the DataFrame
    valid_descendants = [desc for desc in descendants if desc in df.columns]
    if valid_descendants:
        df[f"sum_descendants_{ancestor}"] = df[valid_descendants].sum(axis=1)
# Convert new columns to DataFrame and concatenate
new_df = pd.DataFrame(new_columns)
df = pd.concat([df.drop(columns=["correct_intent"]), new_df], axis=1)

# Save to new CSV
df.to_csv("updated_e5-base_train_banking_trial_transformer.csv", index=False)
print(df.head())


                                       user_question  activate_my_card  \
0                     I am still waiting on my card?          0.001528   
1  What can I do if my card still hasn't arrived ...          0.001406   
2  I have been waiting over a week. Is the card s...          0.001401   
3  Can I track my card while it is in the process...          0.001720   
4  How do I know if I will get my card, or if it ...          0.005147   

   card_about_to_expire  card_arrival  card_delivery_estimate  \
0              0.001989      0.883417                0.011511   
1              0.002514      0.879794                0.011035   
2              0.003467      0.864265                0.023012   
3              0.002178      0.884329                0.015154   
4              0.006411      0.530503                0.014563   

   card_not_working  card_swallowed  get_physical_card  getting_spare_card  \
0          0.008209        0.003444           0.002382            0.001123   
1       

In [ ]:
import pandas as pd
from sklearn.metrics import f1_score

# Load the dataset
df = pd.read_csv("updated_e5-base_train_banking_trial_SVM.csv")



# Dictionary to store optimal thresholds
optimal_thresholds = {}

# Iterate over each intent
for intent in intents:
    # Extract belief values and ground truth
    belief_values = df[intent]
    ground_truth = df[f"is_correct_{intent}"]

    # Initialize variables to track the best threshold and weighted F1-score
    best_threshold = 0.0
    best_weighted_f1 = 0.0

    # Perform grid search over possible thresholds
    for threshold in [i * 0.01 for i in range(101)]:  # From 0.0 to 1.0 in steps of 0.01
        # Predict based on the threshold
        predictions = (belief_values >= threshold).astype(int)

        # Compute weighted F1-score
        weighted_f1 = f1_score(ground_truth, predictions, average="weighted")
        #print(f"Intent: {intent}, Threshold: {threshold}, Weighted F1-Score: {weighted_f1}")
        # Update the best threshold if this weighted F1-score is better
        if weighted_f1 > best_weighted_f1:
            best_weighted_f1 = weighted_f1
            best_threshold = threshold

    # Store the optimal threshold for this intent
    optimal_thresholds[intent] = best_threshold
# Print the optimal thresholds
print("Optimal Thresholds (Weighted F1-Score):")
for intent, threshold in optimal_thresholds.items():
    print(f"{intent}: {threshold:.2f}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score

# Load the dataset
df = pd.read_csv("updated_e5-base_train_banking_trial_transformer.csv")

# Generate list of all intents from the columns
intents = sorted([col.replace("is_correct_", "") for col in df.columns if col.startswith("is_correct_")])

# Dictionary to store optimal thresholds and scores
optimal_results = {}

# Precompute thresholds from 0.00 to 1.00
thresholds = np.linspace(0, 1, 101)

# Iterate over each intent
for intent in intents:
    belief_values = df[intent].values
    ground_truth = df[f"is_correct_{intent}"].values

    best_threshold = 0.0
    best_weighted_f1 = 0.0

    for threshold in thresholds:
        predictions = (belief_values >= threshold).astype(int)
        weighted_f1 = f1_score(ground_truth, predictions, average="weighted")

        if weighted_f1 > best_weighted_f1:
            best_weighted_f1 = weighted_f1
            best_threshold = threshold

    optimal_results[intent] = {
        "threshold": best_threshold,
        "f1_score": best_weighted_f1
    }

# Print the optimal thresholds and scores
print("Optimal Thresholds and Weighted F1 Scores:")
for intent, result in optimal_results.items():
    print(f"{intent}: Threshold = {result['threshold']:.2f}, F1 Score = {result['f1_score']:.4f}")


Optimal Thresholds and Weighted F1 Scores:
Account Management: Threshold = 0.52, F1 Score = 0.9961
Account Services: Threshold = 0.43, F1 Score = 0.9941
Card Issues: Threshold = 0.40, F1 Score = 0.9953
Card Payments: Threshold = 0.35, F1 Score = 0.9895
Cash Withdrawals: Threshold = 0.37, F1 Score = 0.9977
Charges and Fees: Threshold = 0.22, F1 Score = 0.9989
Fees and Refunds: Threshold = 0.24, F1 Score = 0.9982
General: Threshold = 0.45, F1 Score = 0.9976
General Inquiries: Threshold = 0.45, F1 Score = 0.9976
Payment Problems: Threshold = 0.35, F1 Score = 0.9895
Payments and Transfers: Threshold = 0.40, F1 Score = 0.9913
Physical Cards: Threshold = 0.33, F1 Score = 0.9964
Refund_not_showing_up: Threshold = 0.14, F1 Score = 0.9996
Refunds: Threshold = 0.23, F1 Score = 0.9993
Security Issues: Threshold = 0.26, F1 Score = 0.9990
Security and Verification: Threshold = 0.28, F1 Score = 0.9986
Top-Ups: Threshold = 0.21, F1 Score = 0.9970
Transfers: Threshold = 0.37, F1 Score = 0.9938
Verific

In [ ]:
# Save the optimal thresholds and scores to a JSON file
with open("optimal_thresholds_modified_e5-base_banking_transformer.json", "w") as json_file:
    json.dump(optimal_results, json_file, indent=4)

# Print the saved thresholds and scores
print("Optimal Thresholds and Weighted F1 Scores saved to 'optimal_thresholds.json':")
for intent, result in optimal_results.items():
    print(f"{intent}: Threshold = {result['threshold']:.2f}")


Optimal Thresholds and Weighted F1 Scores saved to 'optimal_thresholds.json':
Account Management: Threshold = 0.52
Account Services: Threshold = 0.43
Card Issues: Threshold = 0.40
Card Payments: Threshold = 0.35
Cash Withdrawals: Threshold = 0.37
Charges and Fees: Threshold = 0.22
Fees and Refunds: Threshold = 0.24
General: Threshold = 0.45
General Inquiries: Threshold = 0.45
Payment Problems: Threshold = 0.35
Payments and Transfers: Threshold = 0.40
Physical Cards: Threshold = 0.33
Refund_not_showing_up: Threshold = 0.14
Refunds: Threshold = 0.23
Security Issues: Threshold = 0.26
Security and Verification: Threshold = 0.28
Top-Ups: Threshold = 0.21
Transfers: Threshold = 0.37
Verification: Threshold = 0.22
Virtual Cards: Threshold = 0.44
activate_my_card: Threshold = 0.21
age_limit: Threshold = 0.23
apple_pay_or_google_pay: Threshold = 0.08
atm_support: Threshold = 0.22
automatic_top_up: Threshold = 0.09
balance_not_updated_after_bank_transfer: Threshold = 0.40
balance_not_updated_aft

In [ ]:
import json

# Load the original JSON structure
with open('optimal_thresholds_modified_e5-base_banking_transformer.json', 'r') as f:
    full_data = json.load(f)

# Extract the simplified format
simple_thresholds = {intent: data["threshold"] for intent, data in full_data.items()}


# Print a few to preview
print(dict(list(simple_thresholds.items())))


{'Account Management': 0.52, 'Account Services': 0.43, 'Card Issues': 0.4, 'Card Payments': 0.35000000000000003, 'Cash Withdrawals': 0.37, 'Charges and Fees': 0.22, 'Fees and Refunds': 0.24, 'General': 0.45, 'General Inquiries': 0.45, 'Payment Problems': 0.35000000000000003, 'Payments and Transfers': 0.4, 'Physical Cards': 0.33, 'Refund_not_showing_up': 0.14, 'Refunds': 0.23, 'Security Issues': 0.26, 'Security and Verification': 0.28, 'Top-Ups': 0.21, 'Transfers': 0.37, 'Verification': 0.22, 'Virtual Cards': 0.44, 'activate_my_card': 0.21, 'age_limit': 0.23, 'apple_pay_or_google_pay': 0.08, 'atm_support': 0.22, 'automatic_top_up': 0.09, 'balance_not_updated_after_bank_transfer': 0.4, 'balance_not_updated_after_cheque_or_cash_deposit': 0.54, 'beneficiary_not_allowed': 0.23, 'cancel_transfer': 0.75, 'card_about_to_expire': 0.22, 'card_acceptance': 0.06, 'card_arrival': 0.53, 'card_delivery_estimate': 0.22, 'card_linking': 0.41000000000000003, 'card_not_working': 0.16, 'card_payment_fee_c

In [ ]:
import pandas as pd
import json

def get_leaf_intents(node):
    leaves = []
    if isinstance(node, dict):
        for v in node.values():
            leaves.extend(get_leaf_intents(v))
    elif isinstance(node, list):
        for item in node:
            leaves.extend(get_leaf_intents(item))
    else:
        leaves.append(node)
    return leaves


leaf_intents = set(get_leaf_intents(hierarchy))

# Load your CSV
df = pd.read_csv('updated_train_clinc_trial.csv')

# Extract column names
intent_cols = [col for col in df.columns if col in leaf_intents]
correct_cols = [col for col in df.columns if col.startswith('is_correct_')]

# Prepare output rows
output_rows = []

for _, row in df.iterrows():
    # 1. Predicted intent = leaf with highest belief score
    leaf_scores = {intent: row[intent] for intent in intent_cols}
    predicted_intent = max(leaf_scores.items(), key=lambda x: x[1])[0]

    # 2. Actual intent = leaf where is_correct_<intent> is 1
    correct_intents = [col.replace("is_correct_", "") for col in correct_cols if row[col] == 1 and col.replace("is_correct_", "") in leaf_intents]
    actual_intent = correct_intents[0] if correct_intents else None

    output_rows.append({
        "user_question": row["user_question"],
        "predicted_leaf_intent": predicted_intent,
        "actual_leaf_intent": actual_intent
    })

# Save to new CSV
output_df = pd.DataFrame(output_rows)
output_df.to_csv('intent_predictions_vs_actuals.csv', index=False)


In [ ]:
import pandas as pd

# Load your CSV with predicted and actual intents
df = pd.read_csv("intent_predictions_vs_actuals.csv")

# Compare predicted and actual intents
correct_predictions = (df['predicted_leaf_intent'] == df['actual_leaf_intent']).sum()

# Calculate total number of rows
total_predictions = len(df)

# Calculate accuracy
accuracy = correct_predictions / total_predictions if total_predictions else 0

# Print the accuracy
print(f"Accuracy: {accuracy:.4f}")


Accuracy: 0.6046


In [ ]:
import pandas as pd

# Your hierarchy data (replace this with the actual hierarchy data you provided)

# Create the leaf_to_parent mapping
leaf_to_parent = {}

def build_leaf_to_parent_mapping(hierarchy):
    for parent, children in hierarchy.items():
        for leaf in children:
            leaf_to_parent[leaf] = parent

# Populate the leaf_to_parent dictionary
build_leaf_to_parent_mapping(hierarchy)

# Load the CSV with predicted and actual leaf intents
df = pd.read_csv("intent_predictions_vs_actuals.csv")

# Function to get the parent intent for a leaf intent
def get_parent(leaf_intent):
    return leaf_to_parent.get(leaf_intent, None)  # Return None if no parent found

# Get the parent for both predicted and actual leaf intents
df['predicted_parent'] = df['predicted_leaf_intent'].apply(get_parent)
df['actual_parent'] = df['actual_leaf_intent'].apply(get_parent)

# Compare if the parent intents match
df['parent_accuracy'] = df['predicted_parent'] == df['actual_parent']

# Calculate parent-level accuracy
correct_parent_predictions = df['parent_accuracy'].sum()
total_predictions = len(df)

parent_accuracy = correct_parent_predictions / total_predictions if total_predictions else 0

# Print the parent-level accuracy
print(f"Parent-level Accuracy: {parent_accuracy:.4f}")

# Optionally, save the results to a new CSV file
df.to_csv("predicted_vs_actual_with_parent_accuracy.csv", index=False)


Parent-level Accuracy: 0.7886


In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score

# Load Banking77 dataset
dataset = load_dataset("banking77")
train_data = dataset["train"]
test_data = dataset["test"]

# No OOS filtering needed (Banking77 has only in-scope intents)
train_in_scope = train_data
test_in_scope = test_data

# Load Sentence-BERT
model = SentenceTransformer('intfloat/e5-base')

# Generate embeddings
train_texts = [ex["text"] for ex in train_in_scope]
test_texts = [ex["text"] for ex in test_in_scope]
#train_embeddings = model.encode(train_texts, batch_size=64, show_progress_bar=True)
test_embeddings = model.encode(test_texts, batch_size=64, show_progress_bar=True)

# Get intent labels (convert to label indices)
train_labels = [ex["label"] for ex in train_in_scope]
test_labels = [ex["label"] for ex in test_in_scope]



Batches:   0%|          | 0/49 [00:00<?, ?it/s]

In [ ]:
import pandas as pd

# File names for the chunks
chunk_files = [
    "predictions_test_chunk_0_e5-base_banking_temp=0.05.csv",
    "predictions_test_chunk_1_e5-base_banking_temp=0.05.csv",
    "predictions_test_chunk_2_e5-base_banking_temp=0.05.csv",
    "predictions_test_chunk_3_e5-base_banking_temp=0.05.csv",
    "predictions_test_chunk_4_e5-base_banking_temp=0.05.csv",
    "predictions_test_chunk_5_e5-base_banking_temp=0.05.csv",
    "predictions_test_chunk_6_e5-base_banking_temp=0.05.csv",
    "predictions_test_chunk_7_e5-base_banking_temp=0.05.csv",
    "predictions_test_chunk_8_e5-base_banking_temp=0.05.csv",
    "predictions_test_chunk_9_e5-base_banking_temp=0.05.csv",
]

# Load and concatenate all chunks
dfs = [pd.read_csv(file) for file in chunk_files]
merged_df = pd.concat(dfs, ignore_index=True)

# Save merged file (optional but useful)
merged_df.to_csv("predictions_test_e5-base_banking_temp=0.05.csv", index=False)

print("Merged all chunks into _clinc_trial.csv")

Merged all chunks into _clinc_trial.csv


In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from statsmodels.stats.contingency_tables import mcnemar



# Load the dataset from the CSV file (the one you mentioned earlier)
data = pd.read_csv('predictions_test_e5-base_banking_temp=0.05.csv')

# Extract true labels and model predictions
true_labels = data['true_intent'].values
print(true_labels[0])
# Manually encode the true labels to numeric values using the label_to_int dictionary

# Extract your model's predictions from the CSV (first element of the tuple in 'prediction' column)
model_predictions = data['prediction'].apply(lambda x: eval(x)[0]).values
print(model_predictions[0])
# Manually encode the model predictions to numeric values using the label_to_int dictionary
# Make predictions using the logistic model
logistic_predictions = clf.predict(test_embeddings)
print(logistic_predictions[0])

# Create the contingency table for the McNemar test
# b = Number of instances where SVM is correct and your model is wrong
# c = Number of instances where your model is correct and SVM is wrong
b = np.sum((logistic_predictions == true_labels) & (model_predictions!= true_labels))
c = np.sum((logistic_predictions != true_labels) & (model_predictions == true_labels))

# Construct the contingency table
contingency_table = np.array([[0, b], [c, 0]])

# Perform McNemar test
result = mcnemar(contingency_table, exact=True)

# Print the result
print(f"McNemar test result: statistic={result.statistic}, p-value={result.pvalue}")


card_arrival
card_linking
card_linking
McNemar test result: statistic=61.0, p-value=0.0026347303601547537


In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Initialize the SVM classifier with probability estimation enabled
svm_model = SVC(kernel='linear', probability=True, random_state=42)

# Train the SVM model using the embeddings and labels
svm_model.fit(train_embeddings, train_labels)

# Make predictions (probabilities) on the test data
test_preds_proba = svm_model.predict_proba(test_embeddings)

# Optionally, you can get the predicted labels (based on max probability)
test_preds = svm_model.predict(test_embeddings)

# Evaluate accuracy based on predicted labels
test_accuracy = accuracy_score(test_labels, test_preds)
print(f"Test Accuracy: {test_accuracy}")


Test Accuracy: 0.9146103896103897


In [ ]:
import pickle

# Save model
with open('banking77_XGB_model_e5-base_SVM_0.915.pkl', 'wb') as f:
    pickle.dump(svm_model, f)




In [ ]:
probs = clf.predict_proba(test_embeddings)


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Recreate encoder from the original training labels
train_labels_str = [ex["label"] for ex in train_in_scope]
encoder = LabelEncoder()
encoder.fit(train_labels_str)


LabelEncoder()

In [ ]:
from datasets import load_dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
from sklearn.metrics import accuracy_score
import numpy as np
import torch
from transformers import TrainingArguments
# Load dataset
dataset = load_dataset("banking77")


all_labels = set()
for split in dataset.values():
    for example in split:
        all_labels.add(example["label"])
labels = sorted(all_labels)
# Create label mapping
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}


# Load tokenizer and model
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# Preprocessing function
def preprocess(example):
    tokenized = tokenizer(example["text"], truncation=True)
    tokenized["label"] = label2id[example["label"]]
    return tokenized

# Apply preprocessing
encoded_dataset = dataset.map(preprocess, batched=False)
encoded_dataset.set_format("torch")

# Data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Metric
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    save_steps=500,
    eval_steps=500,
    save_total_limit=3,
    report_to="none"  # Disable logging to W&B
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Train
trainer.train()

# Evaluate
results = trainer.evaluate()
print(f"Test Accuracy: {results['eval_accuracy']:.4f}")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/14.4k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/298k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/93.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10003 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/10003 [00:00<?, ? examples/s]

Map:   0%|          | 0/3080 [00:00<?, ? examples/s]

<ipython-input-2-e713699610e9>:71: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
10,4.371700
20,4.332500
30,4.295300
40,4.280200
50,4.272300
60,4.197300
70,4.090700
80,3.899500
90,3.952700
100,3.702000


Test Accuracy: 0.9075


In [ ]:
# Evaluate
results = trainer.evaluate()
print(f"Test Accuracy: {results['eval_accuracy']:.4f}")

Test Accuracy: 0.9075
